<a href="https://colab.research.google.com/github/rajuchicha52/qua/blob/main/q0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install qiskit qiskit-machine-learning --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.9 MB/s eta 0:00:00


In [4]:
!pip install pylatexenc matplotlib opencv-python faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=b51307ed38cb9ecbddb6a3636d5580dabda848df8775feb570646d6d64b96afa
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [5]:
import os

dataset_path = '/content/drive/MyDrive/SOCOFing/Real'

print("Path exists:", os.path.exists(dataset_path))

Path exists: True


In [6]:
files = os.listdir(dataset_path)
print("Total files:", len(files))
print("Sample files:", files[:10])

Total files: 6000
Sample files: ['544__M_Right_thumb_finger.BMP', '563__M_Left_little_finger.BMP', '52__M_Right_little_finger.BMP', '561__M_Left_middle_finger.BMP', '531__M_Right_middle_finger.BMP', '540__F_Left_index_finger.BMP', '536__F_Left_ring_finger.BMP', '546__M_Right_little_finger.BMP', '558__M_Right_ring_finger.BMP', '547__M_Left_middle_finger.BMP']


In [7]:
import cv2
import numpy as np
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [8]:
def get_label(filename):
    return filename.split("__")[0]

In [9]:
from torchvision import transforms

def apply_clahe(img):
    img_np = np.array(img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_clahe = clahe.apply(img_np)
    return Image.fromarray(img_clahe)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(apply_clahe),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485], std=[0.229])  # for grayscale
])

In [10]:
import random

class SiameseFingerprintDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = [f for f in os.listdir(root_dir) if f.endswith(".BMP")]
        # Group images by label
        self.label_to_images = {}
        for file in self.image_paths:
            label = get_label(file)
            if label not in self.label_to_images:
                self.label_to_images[label] = []
            self.label_to_images[label].append(file)

        self.labels = list(self.label_to_images.keys())

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img1_name = self.image_paths[idx]
        label1 = get_label(img1_name)

        # Decide same or different pair
        if random.random() > 0.5:
            # SAME pair
            img2_name = random.choice(self.label_to_images[label1])
            label = 1
        else:
            # DIFFERENT pair
            label2 = random.choice(self.labels)
            while label2 == label1:
                label2 = random.choice(self.labels)
            img2_name = random.choice(self.label_to_images[label2])
            label = 0

        # Load images
        img1 = Image.open(os.path.join(self.root_dir, img1_name)).convert("L")
        img2 = Image.open(os.path.join(self.root_dir, img2_name)).convert("L")
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, torch.tensor(label, dtype=torch.float32)

In [11]:
from torch.utils.data import random_split

dataset_path = '/content/drive/MyDrive/SOCOFing/Real'
dataset = SiameseFingerprintDataset(dataset_path, transform=transform)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"Train Size: {len(train_dataset)} | Test Size: {len(test_dataset)}")


In [12]:
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Updated way
resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)
for name, param in resnet.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False
    param.requires_grad = False

# Modify for grayscale
resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Output → 12 features
resnet.fc = nn.Linear(512, 12)

resnet = resnet.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


In [13]:
class HybridQNNModel(nn.Module):
    def __init__(self, base_model, q_layer):
        super().__init__()
        self.base = base_model
        self.qnn = q_layer
    
    def forward(self, x1, x2):
        f1 = self.base(x1)
        f2 = self.base(x2)
        combined = torch.cat([f1, f2], dim=1)
        raw = self.qnn(combined)
        F = (raw.squeeze() / 4 + 1) / 2
        return F


In [16]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

SiameseNet(
  (base): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_run

In [18]:
num_qubits = 4
qc = QuantumCircuit(num_qubits)

input_params = [Parameter(f"x{i}") for i in range(24)]
weights = [Parameter(f"w{i}") for i in range(12)]

# Encoding Image A
for i in range(4):
    qc.rx(input_params[3*i], i)
    qc.ry(input_params[3*i+1], i)
    qc.rz(input_params[3*i+2], i)

# Entanglement
qc.cx(0,1)
qc.cx(1,2)
qc.cx(2,3)

# Encoding Image B (Data Re-uploading)
for i in range(4):
    qc.rx(input_params[12 + 3*i], i)
    qc.ry(input_params[12 + 3*i+1], i)
    qc.rz(input_params[12 + 3*i+2], i)

# Entanglement 2
qc.cx(0,1)
qc.cx(1,2)
qc.cx(2,3)

# Trainable layer
for i in range(4):
    qc.ry(weights[3*i], i)
    qc.rz(weights[3*i+1], i)
    qc.rx(weights[3*i+2], i)

In [19]:
observable = SparsePauliOp.from_list([
    ("ZIII", 1),
    ("IZII", 1),
    ("IIZI", 1),
    ("IIIZ", 1),
])

In [20]:
qnn = EstimatorQNN(
    circuit=qc,
    observables=observable,
    input_params=input_params,
    weight_params=weights,
    input_gradients=True
)

In [21]:
quantum_layer = TorchConnector(qnn)
qc.draw("mpl")


In [24]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
    def forward(self, similarity, label):
        loss_sim = label * (1 - similarity)**2
        loss_diff = (1 - label) * torch.clamp(similarity - self.margin, min=0)**2
        return torch.mean(loss_sim + loss_diff)

criterion = ContrastiveLoss(margin=0.3)


In [25]:
optimizer = torch.optim.Adam([
    {'params': model.qnn.parameters(), 'lr': 0.005}, # High LR for Quantum
    {'params': model.base.parameters(), 'lr': 0.0001} # Standard LR for CNN
], weight_decay=1e-5)


In [ ]:
from tqdm.auto import tqdm

# Updated Training Loop with Differential Learning Rates
MAX_STEPS = 120 # Increased density
epochs = 10     # Increased patience
best_loss = float('inf')
epoch_losses = []

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    step = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", total=MAX_STEPS)
    for img1, img2, label in pbar:
        if step >= MAX_STEPS: break

        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        
        optimizer.zero_grad()
        F = model(img1, img2)
        loss = criterion(F, label)
        
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        step += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / step
    epoch_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), '/content/drive/MyDrive/SOCOFing/hybrid_qnn_best.pth')
        print("   >> Saved new best model!")


In [ ]:
# === Loss Curve Visualization ===
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(epoch_losses)+1), epoch_losses, marker='o', color='royalblue', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Average Contrastive Loss')
plt.title('Hybrid Quantum-Classical Training Loss Curve')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
print(f"Best Loss Achieved: {min(epoch_losses):.4f} at Epoch {epoch_losses.index(min(epoch_losses))+1}")


In [ ]:
import faiss
import torch.nn.functional as F_torch
import time

# Build 12D Vector Database using FAISS
model.eval()

gallery_embeddings = []
gallery_labels = []

print("Building FAISS Database...")
with torch.no_grad():
    for img1, _, label in test_loader:
        img1 = img1.to(device)
        emb1 = model.base(img1)
        gallery_embeddings.append(emb1.cpu().numpy())
        
gallery_embeddings = np.concatenate(gallery_embeddings, axis=0)
index = faiss.IndexFlatL2(12)
index.add(gallery_embeddings)

print(f"Database initialized with {index.ntotal} templates.")

# Simulating 1:N Verification
query_emb = gallery_embeddings[0:1]
K = 3
distances, indices = index.search(query_emb, K)
print(f"Top-{K} FAISS classical candidates: {indices[0]}")

# Perform Quantum Rescoring
best_score = 0
best_candidate = -1
for idx in indices[0]:
    cand_emb = gallery_embeddings[idx:idx+1]
    # Manual rescoring for candidates
    q_input = torch.cat([
        torch.tensor(query_emb, dtype=torch.float32), 
        torch.tensor(cand_emb, dtype=torch.float32)
    ], dim=1).to(device)
    
    raw = quantum_layer(q_input)
    q_score = ((raw.squeeze() / 4 + 1) / 2).item()
    print(f"Quantum Match Score against candidate {idx}: {q_score:.4f}")
    if q_score > best_score:
        best_score = q_score
        best_candidate = idx
        
print(f"\nFinal Match identified as Candidate ID {best_candidate} with Quantum Score: {best_score:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from scipy.optimize import brentq
from scipy.interpolate import interp1d

model.eval()
all_scores = []
all_labels = []

print("Evaluating Quantum Siamese Network on Validation Batch...")
with torch.no_grad():
    for img1, img2, label in test_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        F = model(img1, img2)
        all_scores.extend(F.cpu().numpy().flatten())
        all_labels.extend(label.cpu().numpy().flatten())

# Compute False Positive Rates and True Positive Rates
fpr, tpr, thresholds = roc_curve(all_labels, all_scores)
roc_auc = auc(fpr, tpr)

# Compute Equal Error Rate (EER)
eer = brentq(lambda x : 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
thresh = interp1d(fpr, thresholds)(eer)

# Plot ROC Curve
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.scatter([eer], [1-eer], marker='D', color='red', label=f'EER = {eer:.3f} (Thresh={thresh:.3f})', zorder=5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (Imposter Acceptance)')
plt.ylabel('True Positive Rate (Genuine Acceptance)')
plt.title('Receiver Operating Characteristic - Hybrid Quantum Model')
plt.legend(loc="lower right")
plt.show()

print(f"\nFINAL EVALUATION: AUC: {roc_auc:.4f} | Equal Error Rate (EER): {eer:.4f}")

# Binarize predictions using the calculated EER threshold
optimal_thresh = thresh
binary_preds = (np.array(all_scores) >= optimal_thresh).astype(int)
binary_labels = np.array(all_labels).astype(int)

# Compute Matrix Metrics
cm = confusion_matrix(binary_labels, binary_preds)
acc = accuracy_score(binary_labels, binary_preds)
prec = precision_score(binary_labels, binary_preds, zero_division=0)
rec = recall_score(binary_labels, binary_preds, zero_division=0)
f1 = f1_score(binary_labels, binary_preds, zero_division=0)

print("\n--- Accuracy & Confusion Matrix ---")
print(f"Optimal Match Threshold: {optimal_thresh:.4f}")
print(f"Accuracy:  {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall:    {rec*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")
print("\nConfusion Matrix:")
print("[ TN , FP ]")
print("[ FN , TP ]")
print(cm)
